In [1]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

# Đọc dữ liệu gốc (có cột _label)
df = pd.read_csv('../data/processed/german_clean.csv')

print("Shape:", df.shape)
print("\nCác cột _label (dùng cho luật kết hợp):")
label_cols = [c for c in df.columns if c.endswith('_label')]
print(label_cols)
print(f"\nSố cột label: {len(label_cols)}")

Shape: (1000, 35)

Các cột _label (dùng cho luật kết hợp):
['checking_status_label', 'credit_history_label', 'purpose_label', 'savings_label', 'employment_label', 'personal_status_label', 'other_parties_label', 'property_label', 'other_plans_label', 'housing_label', 'job_label', 'telephone_label', 'foreign_worker_label', 'credit_risk_label']

Số cột label: 14


In [2]:
# Chọn các cột label + credit_risk_label để đưa vào luật kết hợp
# Bỏ các cột _label không cần thiết (giữ lại các biến quan trọng)
cols_for_rules = [
    'checking_status_label', 'credit_history_label', 'purpose_label',
    'savings_label', 'employment_label', 'personal_status_label',
    'other_parties_label', 'property_label', 'housing_label',
    'job_label', 'foreign_worker_label', 'credit_risk_label'
]

# Tạo DataFrame one-hot: mỗi giá trị thành một cột bool
df_rules = pd.get_dummies(df[cols_for_rules], dtype=bool)

print("Shape sau one-hot:", df_rules.shape)
print("\n5 cột đầu tiên:")
print(df_rules.columns[:5].tolist())
print("\n5 cột cuối cùng:")
print(df_rules.columns[-5:].tolist())
print(f"\nSố cột: {df_rules.shape[1]}")

Shape sau one-hot: (1000, 51)

5 cột đầu tiên:
['checking_status_label_0-200 DM', 'checking_status_label_< 0 DM', 'checking_status_label_>= 200 DM', 'checking_status_label_no account', 'credit_history_label_all paid (this bank)']

5 cột cuối cùng:
['job_label_unskilled resident', 'foreign_worker_label_no', 'foreign_worker_label_yes', 'credit_risk_label_Bad', 'credit_risk_label_Good']

Số cột: 51


In [3]:
# Chạy Apriori với min_support=0.1 (xuất hiện trong ít nhất 10% hồ sơ)
frequent_itemsets = apriori(df_rules, min_support=0.1, use_colnames=True)

# Sắp xếp theo support giảm dần
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index(drop=True)

print(f"Số frequent itemsets: {len(frequent_itemsets)}")
print("\n10 itemsets phổ biến nhất:")
print(frequent_itemsets.head(10).to_string(index=False))

Số frequent itemsets: 1317

10 itemsets phổ biến nhất:
 support                                                        itemsets
   0.963                           frozenset({foreign_worker_label_yes})
   0.907                           frozenset({other_parties_label_none})
   0.880 frozenset({foreign_worker_label_yes, other_parties_label_none})
   0.713                                  frozenset({housing_label_own})
   0.700                             frozenset({credit_risk_label_Good})
   0.685        frozenset({housing_label_own, foreign_worker_label_yes})
   0.667   frozenset({credit_risk_label_Good, foreign_worker_label_yes})
   0.647        frozenset({housing_label_own, other_parties_label_none})
   0.635   frozenset({credit_risk_label_Good, other_parties_label_none})
   0.630                frozenset({job_label_skilled employee/official})


In [4]:
# Sinh luật kết hợp từ frequent itemsets
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)

print(f"Số luật ban đầu: {len(rules)}")

# Lọc luật có lift > 1.2 (tương quan dương mạnh)
rules_strong = rules[rules['lift'] > 1.2].sort_values('lift', ascending=False).reset_index(drop=True)

print(f"Số luật sau khi lọc lift > 1.2: {len(rules_strong)}")
print("\n10 luật mạnh nhất (theo lift):")
print(rules_strong[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10).to_string(index=False))

Số luật ban đầu: 4446
Số luật sau khi lọc lift > 1.2: 730

10 luật mạnh nhất (theo lift):
                                                                                        antecedents                                                                                         consequents  support  confidence     lift
frozenset({foreign_worker_label_yes, property_label_unknown/no property, other_parties_label_none})                                                                 frozenset({housing_label_for free})    0.100    0.719424 6.661338
                                                                frozenset({housing_label_for free}) frozenset({foreign_worker_label_yes, property_label_unknown/no property, other_parties_label_none})    0.100    0.925926 6.661338
                                                                frozenset({housing_label_for free})                           frozenset({property_label_unknown/no property, other_parties_label_none})    0.100    0.925926

In [5]:
# Lọc luật có consequent là credit_risk_label_Bad
rules_bad = rules[rules['consequents'].apply(lambda x: 'credit_risk_label_Bad' in x)]

# Sắp xếp theo lift giảm dần
rules_bad = rules_bad.sort_values('lift', ascending=False).reset_index(drop=True)

print(f"Số luật dẫn đến Bad: {len(rules_bad)}")
print("\n15 luật mạnh nhất dẫn đến rủi ro Xấu (Bad):")
print(rules_bad[['antecedents', 'support', 'confidence', 'lift']].head(15).to_string(index=False))

Số luật dẫn đến Bad: 0

15 luật mạnh nhất dẫn đến rủi ro Xấu (Bad):
Empty DataFrame
Columns: [antecedents, support, confidence, lift]
Index: []


In [6]:
# Sinh lại luật với min_threshold thấp hơn để bắt được luật dẫn đến Bad
rules_low = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.3)

print(f"Số luật với confidence >= 0.3: {len(rules_low)}")

# Lọc luật có consequent là credit_risk_label_Bad
rules_bad = rules_low[rules_low['consequents'].apply(lambda x: 'credit_risk_label_Bad' in x)]
rules_bad = rules_bad.sort_values('lift', ascending=False).reset_index(drop=True)

print(f"Số luật dẫn đến Bad: {len(rules_bad)}")
print("\n15 luật mạnh nhất dẫn đến rủi ro Xấu (Bad):")
if len(rules_bad) > 0:
    print(rules_bad[['antecedents', 'support', 'confidence', 'lift']].head(15).to_string(index=False))
else:
    print("Vẫn không có luật nào. Thử hạ min_threshold xuống 0.2.")

Số luật với confidence >= 0.3: 10917
Số luật dẫn đến Bad: 88

15 luật mạnh nhất dẫn đến rủi ro Xấu (Bad):
                                                                                                          antecedents  support  confidence     lift
                        frozenset({foreign_worker_label_yes, checking_status_label_< 0 DM, other_parties_label_none})    0.100    0.436681 2.012356
                                                  frozenset({foreign_worker_label_yes, checking_status_label_< 0 DM})    0.112    0.432432 1.992776
                                                  frozenset({foreign_worker_label_yes, checking_status_label_< 0 DM})    0.100    0.386100 1.980002
                                                  frozenset({checking_status_label_< 0 DM, other_parties_label_none})    0.100    0.420168 1.972620
                                                  frozenset({checking_status_label_< 0 DM, other_parties_label_none})    0.101    0.424370 1.955621
      

In [7]:
print("=" * 60)
print("KẾT LUẬN KHAI PHÁ PHỤ — LUẬT KẾT HỢP")
print("=" * 60)

print(f"\nTổng số luật ban đầu: 4446")
print(f"Số luật sau lọc lift > 1.2: 730")
print(f"Số luật dẫn đến Bad (confidence >= 0.3): 88")

print("\n" + "=" * 60)
print("3 LUẬT MẠNH NHẤT DẪN ĐẾN RỦI RO XẤU")
print("=" * 60)

top3 = rules_bad.head(3)
for i, row in top3.iterrows():
    print(f"\nLuật {i+1}:")
    print(f"  NẾU: {set(row['antecedents'])}")
    print(f"  THÌ: {set(row['consequents'])}")
    print(f"  Support: {row['support']:.3f} ({row['support']*100:.1f}% hồ sơ)")
    print(f"  Confidence: {row['confidence']:.3f} ({row['confidence']*100:.1f}%)")
    print(f"  Lift: {row['lift']:.3f} (gấp {row['lift']:.1f} lần baseline)")

print("\n" + "=" * 60)
print("LUẬT KẾT HỢP PHÁT HIỆN GÌ MÀ LOGISTIC REGRESSION KHÔNG CHỈ RA?")
print("=" * 60)
print("""
1. TỔ HỢP YẾU TỐ: Luật kết hợp chỉ ra rằng sự KẾT HỢP của nhiều yếu tố
   (tài khoản âm + lao động nước ngoài + không bảo lãnh + tiết kiệm ít)
   mạnh hơn từng yếu tố đơn lẻ. Logistic Regression có thể đã học được
   tương tác này qua hệ số, nhưng luật kết hợp diễn giải nó rõ ràng hơn.

2. NGƯỠNG CỤ THỂ: Luật kết hợp cho biết ngưỡng cụ thể (confidence 56.2%)
   — nghĩa là trong 100 hồ sơ có đủ 4 yếu tố trên, 56 hồ sơ là Bad.
   Đây là con số trực quan hơn hệ số log-odds.

3. CHÂN DUNG RỦI RO: Luật kết hợp xác định "chân dung" khách hàng rủi ro cao
   để nhân viên tín dụng có thể rà soát thủ công, bổ sung cho quyết định tự động.
""")

KẾT LUẬN KHAI PHÁ PHỤ — LUẬT KẾT HỢP

Tổng số luật ban đầu: 4446
Số luật sau lọc lift > 1.2: 730
Số luật dẫn đến Bad (confidence >= 0.3): 88

3 LUẬT MẠNH NHẤT DẪN ĐẾN RỦI RO XẤU

Luật 1:
  NẾU: {'foreign_worker_label_yes', 'checking_status_label_< 0 DM', 'other_parties_label_none'}
  THÌ: {'credit_risk_label_Bad', 'savings_label_< 100 DM'}
  Support: 0.100 (10.0% hồ sơ)
  Confidence: 0.437 (43.7%)
  Lift: 2.012 (gấp 2.0 lần baseline)

Luật 2:
  NẾU: {'foreign_worker_label_yes', 'checking_status_label_< 0 DM'}
  THÌ: {'credit_risk_label_Bad', 'savings_label_< 100 DM'}
  Support: 0.112 (11.2% hồ sơ)
  Confidence: 0.432 (43.2%)
  Lift: 1.993 (gấp 2.0 lần baseline)

Luật 3:
  NẾU: {'foreign_worker_label_yes', 'checking_status_label_< 0 DM'}
  THÌ: {'credit_risk_label_Bad', 'other_parties_label_none', 'savings_label_< 100 DM'}
  Support: 0.100 (10.0% hồ sơ)
  Confidence: 0.386 (38.6%)
  Lift: 1.980 (gấp 2.0 lần baseline)

LUẬT KẾT HỢP PHÁT HIỆN GÌ MÀ LOGISTIC REGRESSION KHÔNG CHỈ RA?

1. TỔ